# Assessing sampling and convergence in MD simulations


## Introduction

Previous parts of this workshop have made it clear that concepts like "equilibration" and "convergence" in MD simulations are not always so simple in practice.

In previous notebooks you will have frequently found yourself analysing data which it becomes clear is neither equilibrated or converged, so this notebook will deal with a new data set where - perhaps - that is closer to being the case.

----------

The data comes from a 4 microsecond simulation of chignolin: a small peptide that oscillates between a folded and unfolded state. The simulation comes from a tutorial created by Lillian Chong's group at the University of Pittsburgh for their advanced sampling software [WESTPA](https://westpa.github.io/westpa/) - please checkout this package, it's very useful and well documented!

----

Begin by loading the required Python packages: `numpy` for the maths, `mdtraj` for trajectory analysis, `matplotlib` for the graphs, and `nglview` for structure visualization:

In [ ]:
import numpy as np
import mdtraj as mdt
import nglview as nv
from matplotlib import pyplot as plt
%matplotlib inline

## Part 1: Estimating equilibration from RMSD plots

Let's begin by imagining that rather than having a 4 microsecond simulation of this system, we only had 300 nanoseconds - which maybe sounds like quite a lot anyway, doesn't it?

Let's load that first chunk of data and do what everyone always does to start with - plot the RMSD from the starting structure:

In [ ]:
t = mdt.load('data/chignolin.xtc', top='data/chignolin.pdb')[:1500]
r = mdt.rmsd(t, t[0])
plt.plot(t.time / 1000, r) # convert time from ps to ns
plt.xlabel('time (ns)')
plt.ylabel('rmsd (nanometers)')

It looks like there is some sort of equilibration after 700 picoseconds, the rmsd stabilizing around 0.4-0.5 nm. So it might be tempting to say something like "the simulation has equilibrated after about 700 nanoseconds and the data from 700-1500 nanoseconds can be regarded as converged production-phase data that is valid for analysis. There is no need to run the simulation for any longer."

OK - actually we have 4 microseconds of data - so let's see if this conclusion is valid:

In [ ]:
t = mdt.load('data/chignolin.xtc', top='data/chignolin.pdb')
r = mdt.rmsd(t, t[0])
plt.plot(t.time / 1000, r)
plt.xlabel('time (ns)')
plt.ylabel('rmsd (nanometers)')

It's very clear that our assumption was wrong. The idea that the RMSD is going to stablize to a more-or-less constant value representative of equilibrium is clearly not valid here. How do we attempt to assess "equilibration" and "convergence" in a situation like this?

## Part 2: Insights from waymarking

We have used **waymarking** in previous notebooks to look at how the trajectory - as a walk through shape space - samples different conformational states. Let's apply it here:

In [ ]:
def waymark(traj, dmax, atom_indices=None, max_waymarks=100):
    '''
    Waymark a trajectory

    Work through the snapshots in a trajectory from first to last, saving as "waymarks" each one encountered
    that is > dmax nanometers RMSD from any previously waymarked snapshot.

    Also return a list of labels - the waymark each frame in the trajectory is closest to.
    '''

    n_waymarks = 0
    rmsds = np.ones((max_waymarks, traj.n_frames)) * 1000.0
    current_waymark = 0
    next_waymark = 1
    while next_waymark > 0 and n_waymarks < max_waymarks:
        if atom_indices is not None:
            rmsd = mdt.rmsd(traj, traj[current_waymark], atom_indices=atom_indices)
        else:
            rmsd = mdt.rmsd(traj, traj[current_waymark])
        rmsds[n_waymarks] = rmsd
        next_waymark = (rmsds.min(axis=0) > dmax).argmax()
        if next_waymark > 0:
            current_waymark = next_waymark
            n_waymarks += 1
    
    rmsdmin = rmsds.min(axis=0)
    if np.any(rmsdmin > dmax):
        print(f'Error - more than {max_waymarks} found.')
        return None, None
    else:
        waymarks = []
        labels = rmsds.argmin(axis=0)
        label_order = []
        for i, l in enumerate(labels):
            if not l in label_order:
                label_order.append(l)
                waymarks.append(i)
        labels = [label_order.index(l) for l in labels]
        return waymarks, labels

dmax = 0.5 # Found by a bit of trial and error...
waymarks, labels = waymark(t, dmax)
print(f'{len(waymarks)} waymarks found')

Lets examine the snapshots at the waymarks, using `nglviewer` to make a pseudo-trajectory:

In [ ]:
view = nv.show_mdtraj(t[waymarks])
view.add_representation('licorice', 'all')
view

The first waymark is the native, folded state. You will see that the other waymarks correspond to fully or partially unfolded structures.

As previously, let's see how the number of waymarks found increases with the length of the simulation, and how each is sampled over time:

In [ ]:
plt.figure(figsize=(10, 5))
plt.subplot(121)
nwm = [len(set(labels[:i+1])) for i in range(t.n_frames)]
plt.plot(t.time / 1000, nwm)
plt.xlabel('time (ns)')
plt.ylabel('number of waymarks')
plt.title('Exploration of shape space')
plt.subplot(122)
plt.plot(t.time / 1000, labels, '*')
plt.xlabel('time (ns)')
plt.ylabel('waymark labels')
plt.title('Waymark sampling')
plt.tight_layout()

The left-hand graph shows us that after about 700 nanoseconds, no new regions of shape space are sampled. This is our first piece of evidence that this simulation may indeed be long enough to be considered converged - but its not enough by itself: it suggests all important regions of shape space have been detected, but we don't know that the relative sampling of them has converged.

The right-hand panel shows that all regions of sampled shape space are repeatedly revisited throughout the simulation, though not with equal frequency. There is no evidence for an irreversible relaxation process - when waymarks visited early on the simulation are never returned to. *All* the simulation data seems "valid".

But is the simulation converged? To check this, we can examine how the relative occupancies of the different regions of shape space changes as the simulation is extended: 

In [ ]:
n_waymarks = len(waymarks)
sample_size_increment = 1000
sample_sizes = np.arange(sample_size_increment, len(labels) + sample_size_increment, sample_size_increment)
occupancies = np.zeros((len(sample_sizes), n_waymarks))
# Loop over increasing proportions of the simulation:
for isample, sample_size in enumerate(sample_sizes):
    sample = labels[:sample_size]
    for iwaymark in range(n_waymarks):
        occupancies[isample, iwaymark] = sample.count(iwaymark) / sample_size

# Now plot the results:
for iwaymark in range(n_waymarks):
    plt.plot(sample_sizes // 5, occupancies[:, iwaymark])
    plt.xlabel('time (ns)')
    plt.ylabel('waymark occupancy')

Each line on the graph is the relative occupancy of each of the waymarked regions as a function of the length of the simulation.
It shows that by about 3 microseconds, the relative occupancies of the different regions of shape space have more-or-less converged. The native folded state (blue) is occupied about 45% of the time, two distinct partially unfolded states are occupied about 25% and 15% of the time, and other minor conformational states make up the rest.

## Summary

It can take a long time for a molecular dynamics simulation to generate a good sample of equilibrated, converged, data from which hopefully valid information can be extracted, but it is possible! 

Looking at RMSD plots is seldom satisfactory, but it's not much more effort to use a method such as waymarking, (or for a bit more sophistication and rigour, clustering) which is much more reliable and insightful.